# 26 Local Feature Engineering
Local-only deterministic feature engineering for Bluesky engagement prediction.


**Notebook purpose:** Joins prepared posts, trend matches, hydrated engagement metrics, and actor profiles into a single feature table with engagement labels (LOW/MEDIUM/HIGH) for ML modeling.

**Required data:** `local/derived/bluesky/bluesky_posts_prepared.parquet` (nb 23), `local/derived/matching/bluesky_post_best_trend_matches.parquet` + full matches (nb 25), plus hydrated posts and actor profile JSONL.gz files from the pipeline.

**Run order:** Run after notebook 25 (matching). Run before notebook 27 (baseline modeling).

## 1) Load Inputs and Inspect Schemas


In [4]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'features' / 'feature_engineering.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/features/feature_engineering.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.feature_engineering import (
    FeatureConfig,
    build_local_engagement_feature_table,
    load_actor_profiles_file,
    load_hydrated_metrics_file,
    select_best_actor_profile_source_file,
    select_best_hydrated_source_file,
)
from src.source_paths import resolve_bluesky_source_root

prepared_path = ROOT / "local/derived/bluesky/bluesky_posts_prepared.parquet"
best_matches_path = ROOT / "local/derived/matching/bluesky_post_best_trend_matches.parquet"
full_matches_path = ROOT / "local/derived/matching/bluesky_post_trend_matches.parquet"

for _p, _label in [
    (prepared_path, "notebook 23 (text prep)"),
    (best_matches_path, "notebook 25 (trend matching)"),
    (full_matches_path, "notebook 25 (trend matching)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"DATA NOT YET AVAILABLE -- run {_label} first.\nMissing: {_p}")

prepared_df = pd.read_parquet(prepared_path)
best_matches_df = pd.read_parquet(best_matches_path)
full_matches_df = pd.read_parquet(full_matches_path)

print("prepared:", len(prepared_df), "rows", len(prepared_df.columns), "cols")
print("best matches:", len(best_matches_df), "rows", len(best_matches_df.columns), "cols")
print("full matches:", len(full_matches_df), "rows", len(full_matches_df.columns), "cols")

prepared: 19999 rows 26 cols
best matches: 17958 rows 28 cols
full matches: 300455 rows 28 cols


In [5]:
print("Prepared columns:", prepared_df.columns.tolist())
print()
print("Best-match columns:", best_matches_df.columns.tolist())
print()
print("Full-match columns:", full_matches_df.columns.tolist())


Prepared columns: ['uri', 'post_created_at', 'source_run_tag', 'text_source', 'raw_capture_run_id', 'raw_captured_at', 'raw_repo_did', 'raw_record_created_at', 'hydrated_capture_run_id', 'hydrated_hydrate_run_id', 'hydrated_author_did', 'hydrated_author_handle', 'hydrated_indexed_at', 'hydrated_hydrated_at', 'raw_source_row_json', 'hydrated_source_row_json', 'post_text_raw', 'post_text_clean', 'post_text_alnum', 'post_token_count', 'post_char_count', 'has_hashtag', 'has_url', 'has_mention', 'has_special_chars', 'has_non_ascii']

Best-match columns: ['candidate_id', 'uri', 'post_created_at', 'post_text_raw', 'post_text_clean', 'candidate_phrase_raw', 'candidate_phrase_clean', 'candidate_phrase_alnum', 'candidate_source_type', 'candidate_rank_in_post', 'trend_id', 'trend_date', 'trend_name_raw', 'trend_name_clean', 'trend_key_no_hash', 'trend_counts', 'trend_num_hours', 'match_stage', 'match_method', 'match_score', 'lexical_score', 'semantic_token_cosine', 'semantic_char_trigram_jaccard'

## 2) Select Local Hydrated and Actor Sources


In [6]:
bluesky_source_root = ROOT / "data"
hydrated_selection = select_best_hydrated_source_file(prepared_df, base_dir=bluesky_source_root)
actor_selection = select_best_actor_profile_source_file(prepared_df, base_dir=bluesky_source_root)

hydrated_selection["selected"], actor_selection["selected"]

({'file_path': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/test_data/raw_interactions/hydrated_posts_top25.jsonl.gz',
  'row_count': 25,
  'unique_uri_count': 25,
  'uri_overlap_count': 25,
  'uri_overlap_rate': 0.0012500625031251563},
 {'file_path': '/Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/test_data/raw_profiles/actor_profiles_top25.jsonl.gz',
  'row_count': 25,
  'unique_did_count': 25,
  'did_overlap_count': 25,
  'did_overlap_rate': 0.0018162005085361425})

In [7]:
hydrated_path = Path(hydrated_selection["selected"]["file_path"])
actor_path = Path(actor_selection["selected"]["file_path"])

hydrated_df = load_hydrated_metrics_file(hydrated_path)
actor_df = load_actor_profiles_file(actor_path)

print("selected hydrated:", hydrated_path)
print("hydrated rows:", len(hydrated_df), "unique uri:", hydrated_df["uri"].nunique())
print("selected actor:", actor_path)
print("actor rows:", len(actor_df), "unique did:", actor_df["did"].nunique())


selected hydrated: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/test_data/raw_interactions/hydrated_posts_top25.jsonl.gz
hydrated rows: 25 unique uri: 25
selected actor: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/test_data/raw_profiles/actor_profiles_top25.jsonl.gz
actor rows: 25 unique did: 25


## 3) Build Feature Table (One Row Per `uri`)


In [8]:
config = FeatureConfig(low_quantile=0.33, high_quantile=0.66)

features_df, summary = build_local_engagement_feature_table(
    prepared_df=prepared_df,
    best_matches_df=best_matches_df,
    full_matches_df=full_matches_df,
    hydrated_df=hydrated_df,
    actor_df=actor_df,
    config=config,
)

features_df = features_df.sort_values(["uri"], kind="stable").reset_index(drop=True)

print("feature rows:", len(features_df), "feature cols:", len(features_df.columns))
summary["labeling"]


feature rows: 19999 feature cols: 92


{'rule': 'fallback_count_bins',
 'low_quantile': 0.33,
 'high_quantile': 0.66,
 'low_threshold': 0.0,
 'high_threshold': 0.0,
 'fallback_reason': 'collapsed_quantile_or_empty_class'}

## 4) Join Coverage and Label Distribution


In [9]:
summary["join_coverage"]


{'prepared_to_best_match': {'total_rows': 19999,
  'matched_rows': 17958,
  'left_only_rows': 2041,
  'coverage_rate': 0.8979448972448623},
 'with_candidate_aggregates': {'total_rows': 19999,
  'matched_rows': 17958,
  'left_only_rows': 2041,
  'coverage_rate': 0.8979448972448623},
 'with_hydrated_metrics': {'total_rows': 19999,
  'matched_rows': 25,
  'left_only_rows': 19974,
  'coverage_rate': 0.0012500625031251563},
 'with_actor_profiles': {'total_rows': 19999,
  'matched_rows': 30,
  'left_only_rows': 19969,
  'coverage_rate': 0.0015000750037501875}}

In [10]:
print("Label distribution:")
print(pd.Series(summary["label_distribution"]))
print()
print("Match stage distribution:")
print(pd.Series(summary["match_stage_distribution"]))


Label distribution:
LOW       19993
MEDIUM        5
HIGH          1
dtype: int64

Match stage distribution:
unmatched       11598
semantic         4555
no_candidate     2041
exact            1134
fuzzy             671
dtype: int64


## 5) Feature Groups and Leakage Guard


In [11]:
feature_groups = summary["feature_groups"]
print("model feature count:", len(feature_groups["model_feature_columns"]))
print("label columns:", feature_groups["label_columns"])
print("debug columns count:", len(feature_groups["debug_columns"]))

leakage_columns = {"eng_like_count", "eng_reply_count", "eng_repost_count", "eng_quote_count", "engagement_total", "engagement_label"}
print()
print("leakage columns in model features:", sorted(leakage_columns.intersection(feature_groups["model_feature_columns"])))


model feature count: 67
label columns: ['engagement_label', 'engagement_total', 'eng_like_count', 'eng_reply_count', 'eng_repost_count', 'eng_quote_count']
debug columns count: 17

leakage columns in model features: []


## 6) Quality Checks and Examples


In [12]:
missing_top = sorted(summary["missingness_by_model_feature"].items(), key=lambda kv: (-kv[1], kv[0]))[:15]
print("Top model-feature missingness:")
for key, value in missing_top:
    print(f"- {key}: {value}")


Top model-feature missingness:
- capture_run_id: 19974
- hydrate_run_id: 19974
- hydrated_at: 19974
- indexed_at: 19974
- like_count: 19974
- quote_count: 19974
- reply_count: 19974
- repost_count: 19974
- actor_created_at: 19969
- actor_indexed_at: 19969
- actor_run_id: 19969
- description: 19969
- display_name: 19969
- followers_count: 19969
- follows_count: 19969


In [13]:
features_df[[
    "uri",
    "engagement_total",
    "engagement_label",
    "match_stage",
    "has_trend_match",
    "candidate_count",
    "matched_candidate_count",
    "post_token_count",
    "post_unique_token_count",
    "actor_followers_count",
]].head(15)


,uri,engagement_total,engagement_label,match_stage,has_trend_match,candidate_count,matched_candidate_count,post_token_count,post_unique_token_count,actor_followers_count
0,at://did:plc:222ocxmd3qfo5pn7juncj5ds/app.bsky...,0.0,LOW,semantic,True,20,1,35,32,2115.0
1,at://did:plc:222p42fegwhwfyrc3gqam76j/app.bsky...,0.0,LOW,semantic,True,9,1,11,11,10387.0
2,at://did:plc:223eaz6uecpa3t63jdlojutp/app.bsky...,0.0,LOW,semantic,True,16,1,10,10,2359.0
3,at://did:plc:22auflhdbiemhxvkgvpusmmd/app.bsky...,0.0,LOW,fuzzy,True,12,1,16,15,47.0
4,at://did:plc:22bb4y5cxdp3eucibibb4fkp/app.bsky...,0.0,LOW,unmatched,False,20,0,21,17,3555.0
5,at://did:plc:22bixok3zcw6dv72gyi5pwox/app.bsky...,0.0,LOW,semantic,True,20,1,17,19,56.0
6,at://did:plc:22bixok3zcw6dv72gyi5pwox/app.bsky...,0.0,LOW,semantic,True,2,1,6,6,56.0
7,at://did:plc:22dulstyhpc7jfpogwtyqkhv/app.bsky...,0.0,LOW,exact,True,20,1,26,25,1045.0
8,at://did:plc:22ezkuvas6f545oal47snp5x/app.bsky...,0.0,LOW,no_candidate,False,0,0,0,0,2634.0
9,at://did:plc:22fyoakkfli6tcd5kmozvvxp/app.bsky...,0.0,LOW,unmatched,False,3,0,5,5,399.0


## 7) Write Outputs


In [14]:
out_dir = ROOT / "local/derived/features"
out_dir.mkdir(parents=True, exist_ok=True)
sample_dir = ROOT / "data/samples"
sample_dir.mkdir(parents=True, exist_ok=True)

full_out = out_dir / "bluesky_engagement_features.parquet"
sample_parquet_out = sample_dir / "bluesky_engagement_features_sample_1000.parquet"
sample_csv_out = sample_dir / "bluesky_engagement_features_sample_1000.csv"
summary_out = out_dir / "bluesky_engagement_feature_summary.json"

features_df.to_parquet(full_out, index=False)
features_df.head(1000).to_parquet(sample_parquet_out, index=False)
features_df.head(1000).to_csv(sample_csv_out, index=False)

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "26_local_feature_engineering",
    "input_paths": {
        "prepared_posts": str(prepared_path),
        "best_matches": str(best_matches_path),
        "full_matches": str(full_matches_path),
        "selected_hydrated_source": str(hydrated_path),
        "selected_actor_source": str(actor_path),
    },
    "hydrated_selection": hydrated_selection,
    "actor_selection": actor_selection,
    "summary": summary,
    "output_paths": {
        "full_features_parquet": str(full_out),
        "sample_features_parquet": str(sample_parquet_out),
        "sample_features_csv": str(sample_csv_out),
    },
}
summary_out.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("wrote", full_out)
print("wrote", sample_parquet_out)
print("wrote", sample_csv_out)
print("wrote", summary_out)

wrote /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/features/bluesky_engagement_features.parquet
wrote /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/bluesky_engagement_features_sample_1000.parquet
wrote /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/data/samples/bluesky_engagement_features_sample_1000.csv
wrote /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/derived/features/bluesky_engagement_feature_summary.json


## 8) Readiness Statement
Phase 26 is complete when this notebook runs end-to-end, outputs are written, and summary metrics are documented in findings.
